# 📖 Notebook 2: News Feed Ranking

A chronological feed is the simplest approach, but it's not always the best experience.  
Users care more about **relevance** than raw time order. This notebook explores how to rank posts.

## Learning Objectives

By the end of this notebook, you'll understand:
- Why chronological feeds don't scale well for engagement
- How to build a simple scoring function for posts
- The concept of **affinity** (how close are two users?)
- How to combine recency, popularity, and affinity into a feed rank

## 🛠️ Setup

Start the infrastructure first:

```bash
cd system-designs/fb-news-feed
docker-compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras
import time
import math
from datetime import datetime, timedelta

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "newsfeed_demo",
    "user": "demo",
    "password": "demo"
}

def get_db():
    return psycopg2.connect(**DB_CONFIG)

# Test connection
try:
    conn = get_db()
    conn.close()
    print("✅ Connected to PostgreSQL")
except Exception as e:
    print(f"❌ Connection failed: {e}")
    print("   Run: docker-compose up -d")

## 🤔 Why Not Just Sort by Time?

A purely chronological feed has problems:

- Your best friend's post from 2 hours ago gets buried under 50 posts from pages you barely care about
- A viral post with 10,000 likes ranks the same as a post nobody interacted with
- High-frequency posters dominate the feed

Facebook switched from chronological to **ranked** feeds in 2009.  
The result? Users engaged **more** because they saw content they actually cared about.

### The Original Facebook EdgeRank Formula

Facebook's original ranking algorithm was called **EdgeRank**:

```
Score = Affinity × Weight × Decay
```

| Factor | What It Means | Example |
|--------|--------------|--------|
| **Affinity** | How close are you to the author? | You interact with Alice's posts often → high affinity |
| **Weight** | How engaging is this type of content? | Photos > links > plain text |
| **Decay** | How old is the post? | Newer posts score higher |

## Step 1: Add Engagement Data

To rank posts, we need signals. Let's add likes and a simple interaction log to our database.

In [ ]:
conn = get_db()
cur = conn.cursor()

# Create tables for engagement signals
cur.execute("""
    CREATE TABLE IF NOT EXISTS likes (
        user_id INTEGER REFERENCES users(id),
        post_id INTEGER REFERENCES posts(id),
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        PRIMARY KEY (user_id, post_id)
    )
""")

# Interaction log: tracks when a user interacts with another user's content
# This is how we measure "affinity" between two users.
cur.execute("""
    CREATE TABLE IF NOT EXISTS interactions (
        id SERIAL PRIMARY KEY,
        actor_id INTEGER REFERENCES users(id),
        target_user_id INTEGER REFERENCES users(id),
        interaction_type VARCHAR(20),
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
""")
cur.execute("CREATE INDEX IF NOT EXISTS idx_interactions_actor ON interactions(actor_id, target_user_id)")

conn.commit()
print("✅ Created likes and interactions tables")

# Seed some engagement data
import random
random.seed(42)  # reproducible results

# Generate likes — some posts are more popular than others
cur.execute("SELECT id FROM posts")
post_ids = [row[0] for row in cur.fetchall()]

like_data = []
for post_id in post_ids:
    # Celebrity posts get more likes
    cur_temp = conn.cursor()
    cur_temp.execute("SELECT author_id FROM posts WHERE id = %s", (post_id,))
    author_id = cur_temp.fetchone()[0]
    
    n_likes = random.randint(10, 40) if author_id > 50 else random.randint(0, 10)
    likers = random.sample(range(1, 51), min(n_likes, 50))
    for liker in likers:
        like_data.append((liker, post_id))

psycopg2.extras.execute_values(
    cur,
    "INSERT INTO likes (user_id, post_id) VALUES %s ON CONFLICT DO NOTHING",
    like_data
)

# Generate interaction history — user 1 interacts a lot with users 2, 3, and celebrity 51
interaction_data = []
for _ in range(30):
    interaction_data.append((1, 2, 'like'))
    interaction_data.append((1, 3, 'like'))
    interaction_data.append((1, 51, 'like'))
for _ in range(5):
    interaction_data.append((1, 10, 'like'))
    interaction_data.append((1, 20, 'like'))

psycopg2.extras.execute_values(
    cur,
    "INSERT INTO interactions (actor_id, target_user_id, interaction_type) VALUES %s",
    interaction_data
)

conn.commit()
print(f"✅ Seeded {len(like_data)} likes and {len(interaction_data)} interactions")
conn.close()

## Step 2: Build the Scoring Function

Our simple ranking score combines three signals:

```
score = (affinity_score × 3.0) + (popularity_score × 1.0) + (recency_score × 2.0)
```

Each component is normalised to a 0–1 range so we can combine them with weights.

In [ ]:
def compute_affinity(user_id: int, author_id: int, conn) -> float:
    """
    How much does this user interact with the author?
    Returns a value between 0 and 1.
    
    Higher = the user likes/comments on this author's posts frequently.
    """
    cur = conn.cursor()
    cur.execute(
        "SELECT COUNT(*) FROM interactions WHERE actor_id = %s AND target_user_id = %s",
        (user_id, author_id)
    )
    count = cur.fetchone()[0]
    # Use logarithmic scaling so it doesn't grow linearly forever
    # 0 interactions → 0.0, 10 interactions → 0.7, 30+ → ~1.0
    return min(1.0, math.log(count + 1) / math.log(31))


def compute_popularity(post_id: int, conn) -> float:
    """
    How popular is this post? (based on likes)
    Returns a value between 0 and 1.
    """
    cur = conn.cursor()
    cur.execute("SELECT COUNT(*) FROM likes WHERE post_id = %s", (post_id,))
    likes = cur.fetchone()[0]
    # 0 likes → 0.0, 10 likes → 0.7, 40+ → ~1.0
    return min(1.0, math.log(likes + 1) / math.log(41))


def compute_recency(post_created_at: datetime) -> float:
    """
    How recent is this post?
    Returns a value between 0 and 1.
    
    Just posted → 1.0, 1 day ago → 0.5, 1 week ago → ~0.0
    """
    age_hours = (datetime.now() - post_created_at).total_seconds() / 3600
    # Exponential decay: halves every 24 hours
    return math.exp(-0.029 * age_hours)  # 0.029 ≈ ln(2)/24


# Demo: show scores for a few posts from user 1's perspective
conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

cur.execute("""
    SELECT p.id, p.content, p.created_at, p.author_id, u.display_name
    FROM posts p
    JOIN users u ON u.id = p.author_id
    WHERE p.author_id IN (2, 3, 10, 51)
    ORDER BY p.created_at DESC
    LIMIT 8
""")
sample_posts = cur.fetchall()

print("📊 Scoring components for user 1's feed:")
print(f"{'Post':>5}  {'Author':<20}  {'Affinity':>8}  {'Popular':>8}  {'Recency':>8}")
print("-" * 60)

for post in sample_posts:
    aff = compute_affinity(1, post["author_id"], conn)
    pop = compute_popularity(post["id"], conn)
    rec = compute_recency(post["created_at"])
    print(f"{post['id']:>5}  {post['display_name']:<20}  {aff:>8.2f}  {pop:>8.2f}  {rec:>8.2f}")

print("\n💡 User 1 interacts a lot with users 2, 3, and celebrity 51 → high affinity")
conn.close()

## Step 3: Ranked Feed

Now let's build a function that fetches a feed and ranks it by our combined score.

In [ ]:
# Weights for each signal — tune these to change ranking behaviour
WEIGHT_AFFINITY  = 3.0  # how much we value "closeness" to the author
WEIGHT_POPULARITY = 1.0  # how much we value raw likes
WEIGHT_RECENCY   = 2.0  # how much we value freshness


def ranked_feed(user_id: int, limit: int = 20) -> list:
    """
    Get a ranked feed for the user.
    
    1. Fetch candidate posts from people the user follows (recent ones)
    2. Score each post
    3. Sort by score descending
    4. Return top N
    """
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    
    # Get candidate posts (last ~200 posts from followees)
    cur.execute("""
        SELECT p.id, p.content, p.created_at, p.author_id,
               u.username, u.display_name
        FROM posts p
        JOIN users u ON u.id = p.author_id
        WHERE p.author_id IN (
            SELECT followee_id FROM follows WHERE follower_id = %s
        )
        ORDER BY p.created_at DESC
        LIMIT 200
    """, (user_id,))
    candidates = cur.fetchall()
    
    # Score each post
    scored = []
    for post in candidates:
        affinity  = compute_affinity(user_id, post["author_id"], conn)
        popularity = compute_popularity(post["id"], conn)
        recency   = compute_recency(post["created_at"])
        
        score = (
            WEIGHT_AFFINITY  * affinity +
            WEIGHT_POPULARITY * popularity +
            WEIGHT_RECENCY   * recency
        )
        scored.append({**post, "score": score,
                       "_aff": affinity, "_pop": popularity, "_rec": recency})
    
    # Sort by score (highest first)
    scored.sort(key=lambda x: x["score"], reverse=True)
    conn.close()
    
    return scored[:limit]


# Show the ranked feed for user 1
feed = ranked_feed(user_id=1, limit=15)

print("📰 Ranked feed for user 1:")
print(f"{'Rank':>4}  {'Score':>6}  {'Author':<22}  {'Aff':>4} {'Pop':>4} {'Rec':>4}  Post")
print("-" * 90)
for i, post in enumerate(feed, 1):
    print(
        f"{i:>4}  {post['score']:>6.2f}  {post['display_name']:<22}  "
        f"{post['_aff']:.1f}  {post['_pop']:.1f}  {post['_rec']:.1f}   "
        f"{post['content'][:35]}"
    )

print("\n💡 Posts from users with high affinity (2, 3, celebrity 51) rank higher!")

## Step 4: Chronological vs Ranked — A Comparison

In [ ]:
# Get chronological feed for comparison
conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
cur.execute("""
    SELECT p.id, p.content, p.created_at, p.author_id,
           u.username, u.display_name
    FROM posts p
    JOIN users u ON u.id = p.author_id
    WHERE p.author_id IN (
        SELECT followee_id FROM follows WHERE follower_id = 1
    )
    ORDER BY p.created_at DESC
    LIMIT 10
""", )
chrono_feed = cur.fetchall()
conn.close()

ranked = ranked_feed(user_id=1, limit=10)

print("📅 Chronological Feed (newest first):")
print("-" * 60)
for post in chrono_feed:
    print(f"  [{post['created_at']:%m-%d %H:%M}] @{post['username']}: {post['content'][:40]}")

print("\n🏆 Ranked Feed (most relevant first):")
print("-" * 60)
for post in ranked:
    print(f"  [score={post['score']:.1f}] @{post['username']}: {post['content'][:40]}")

print("\n💡 Notice how the ranked feed surfaces posts from users you interact with most,")
print("   even if they're not the absolute newest posts.")

## 🎛️ Experiment: Tuning the Weights

Try changing the weights below to see how the feed changes.  
In production, these weights would be learned from user behaviour using machine learning.

In [ ]:
# ✏️ EXPERIMENT: Change these weights and re-run this cell!
WEIGHT_AFFINITY   = 3.0   # try 0.0 to ignore affinity
WEIGHT_POPULARITY = 1.0   # try 5.0 to make viral posts dominate
WEIGHT_RECENCY    = 2.0   # try 10.0 to make it nearly chronological

feed = ranked_feed(user_id=1, limit=10)

print(f"🎛️  Weights: affinity={WEIGHT_AFFINITY}, popularity={WEIGHT_POPULARITY}, recency={WEIGHT_RECENCY}")
print()
for i, post in enumerate(feed, 1):
    print(
        f"  {i:>2}. [score={post['score']:.2f}] @{post['username']:<15} "
        f"aff={post['_aff']:.1f} pop={post['_pop']:.1f} rec={post['_rec']:.1f}"
    )

## 🧹 Cleanup

In [ ]:
conn = get_db()
cur = conn.cursor()
cur.execute("DROP TABLE IF EXISTS likes CASCADE")
cur.execute("DROP TABLE IF EXISTS interactions CASCADE")
conn.commit()
print("🧹 Cleaned up likes and interactions tables")
conn.close()

## 📚 Summary

### Key Takeaways

1. **Chronological feeds** are simple but don't optimise for what users care about
2. **Ranking** combines affinity, popularity, and recency into a single score
3. **Affinity** measures how much you interact with an author — it's the strongest signal
4. **Weights** control the ranking behaviour; in production, ML models learn optimal weights
5. **EdgeRank** was Facebook's original formula; modern systems use deep learning

### Interview Tips

- Mention ranking as an enhancement after the basic feed design
- The formula `score = affinity × weight × decay` is a great one-liner to drop
- Explain that ranking happens **after** candidate retrieval (first get posts, then score them)
- Note that ranking adds latency — caching ranked feeds is important

### Next Up

In **Notebook 3**, we'll dive into **Social Graph Storage** — how to store follow relationships efficiently and answer queries like "who follows me?" and "mutual friends".